# MNIST Handwritten Digit Classification

**Dense Neural Network using TensorFlow/Keras**

This notebook loads MNIST, normalizes images, one-hot encodes labels, trains a fully connected (non-CNN) model for 5 epochs, evaluates it, plots learning curves, saves an H5 model, and verifies loading/prediction.

## Cell 1 — Imports and reproducibility

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Imports successful.")

## Cell 2 — Load MNIST dataset

In [ ]:
from tensorflow.keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()
print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Test images:", x_test.shape)
print("Test labels:", y_test.shape)

## Cell 3 — Preprocess data
Normalize image pixels to `[0, 1]` and one-hot encode the labels for the categorical cross-entropy loss.

In [ ]:
from tensorflow.keras.utils import to_categorical

NUM_CLASSES = 10
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
y_train = to_categorical(y_train, NUM_CLASSES)
y_test = to_categorical(y_test, NUM_CLASSES)

assert x_train.shape == (60000, 28, 28)
assert x_test.shape == (10000, 28, 28)
assert y_train.shape == (60000, NUM_CLASSES)
assert y_test.shape == (10000, NUM_CLASSES)
assert 0.0 <= x_train.min() and x_train.max() <= 1.0
print("Preprocessing complete; image pixels range from", x_train.min(), "to", x_train.max())

## Cell 4 — Preview a few training images

In [ ]:
plt.figure(figsize=(8, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Label: {y_train[i].argmax()}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## Cell 5 — Build the dense Sequential model
This is a fully connected network; it intentionally does not use convolutional layers.

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Flatten, Dense, Dropout

model = Sequential([
    Input(shape=(28, 28)),
    Flatten(),
    Dense(512, activation="relu"),
    Dropout(0.2),
    Dense(256, activation="relu"),
    Dropout(0.2),
    Dense(NUM_CLASSES, activation="softmax")
])
model.summary()

## Cell 6 — Compile model

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
print("Model compiled.")

## Cell 7 — Train for 5 epochs

In [ ]:
EPOCHS = 5
BATCH_SIZE = 128
history = model.fit(
    x_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    shuffle=True,
    verbose=1
)

## Cell 8 — Evaluate on test data

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy * 100:.2f}%")

## Cell 9 — Plot accuracy and loss curves

In [ ]:
epochs_range = range(1, EPOCHS + 1)
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history.history["accuracy"], marker="o", label="Training accuracy")
plt.plot(epochs_range, history.history["val_accuracy"], marker="o", label="Validation accuracy")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history.history["loss"], marker="o", label="Training loss")
plt.plot(epochs_range, history.history["val_loss"], marker="o", label="Validation loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Cell 10 — Save model as H5 and verify the file

In [ ]:
MODEL_PATH = "mnist_dense_model.h5"
model.save(MODEL_PATH)

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(f"Model was not saved at {MODEL_PATH}")
print(f"Saved model: {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / (1024 * 1024):.2f} MB)")

## Cell 11 — Load saved model and test one prediction

In [ ]:
loaded_model = tf.keras.models.load_model(MODEL_PATH)
sample_index = 0
sample = x_test[sample_index:sample_index + 1]
probabilities = loaded_model.predict(sample, verbose=0)[0]
predicted_digit = int(np.argmax(probabilities))
actual_digit = int(y_test[sample_index])
confidence = float(np.max(probabilities))

plt.imshow(x_test[sample_index], cmap="gray")
plt.axis("off")
plt.title(f"Predicted: {predicted_digit} | Actual: {actual_digit}")
plt.show()
print("Predicted digit:", predicted_digit)
print("Actual digit:", actual_digit)
print(f"Confidence: {confidence * 100:.2f}%")
print("Prediction:", "CORRECT" if predicted_digit == actual_digit else "INCORRECT")
print("Notebook run finished.")